# Notebook 2: Reform RDF triples into a property graph

Import the Python library dependencies.

In [11]:
import json
import pathlib
import re
import shutil

from icecream import ic
import kuzu
import maplib
import polars as pl
import watermark

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-11-11T16:12:51.942126-08:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

json     : 2.0.9
watermark: 2.5.0
polars   : 1.29.0
maplib   : 0.17.12
re       : 2.2.1



## SPARQL queries in Maplib

The `thesaurus.ttl` file provides a _semantic graph_, which we need to reform as a _property graph_.
We'll run SPARQL queries in [`Maplib`](https://github.com/DataTreehouse/maplib), where the result sets from these queries become [`Polars` dataframes](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html), which can then be loaded as tables in [`RyuGraph`](https://ryugraph.io/).

First we load the RDF triples into a `Model` and define RDF [_prefix names_](https://www.w3.org/TR/sparql11-query/#prefNames) used to make SPARQL queries more compact.

In [3]:
rdf_model: maplib.Model = maplib.Model()
rdf_model.read(pathlib.Path("thesaurus.ttl"))

PREFIX_NAMES: str = """
PREFIX dc:   <http://purl.org/dc/elements/1.1/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX sz:   <https://github.com/senzing-garage/sz-semantics/wiki/ns#>
"""

Define an accessor function to streamline how to run SPARQL queries in Maplib.

**Note: this is specific to this use case**, since Maplib does not yet appear to have N3 serialization support?

In [5]:
def run_sparql (
    query: str,
    ) -> pl.DataFrame:
    pat = re.compile(r"(\\?\w+)")
    rs: dict = json.loads(rdf_model.query(PREFIX_NAMES + query, return_json = True))
    rows: list = []

    for line in query.split("\n"):
        if line.strip().startswith("SELECT"):
            cols: list[ str] = re.findall(pat, line.replace("SELECT ", ""))

    for row in rs["results"]["bindings"]:
        row_dict: dict = {}

        for name, attr in row.items():
            row_dict[name] = attr["value"]

        rows.append(row_dict)

    return pl.from_dicts(rows).select(cols)

Build a dataframe for the taxonomy nodes.

In [21]:
df_taxo_nodes: pl.DataFrame = run_sparql("""
SELECT ?id ?descrip
WHERE {
  ?id a skos:Concept ;
    skos:definition ?descrip .
}""")

df_taxo_nodes.replace_column(
    0,
    df_taxo_nodes["id"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_taxo_nodes = df_taxo_nodes.with_columns(pl.lit("skos:Concept").alias("class"))

df_taxo_nodes

id,descrip,class
str,str,str
"""sz:DataRecord""","""Data Record is a composite dat…","""skos:Concept"""
"""sz:Entity""","""Entity is anything that can be…","""skos:Concept"""
"""sz:Organization""","""Organization is a social entit…","""skos:Concept"""
"""sz:Person""","""Person represents an individua…","""skos:Concept"""


Build a dataframe for the `sz:Person` nodes from ER.

In [22]:
df_er_person_nodes: pl.DataFrame = run_sparql("""
SELECT ?id ?descrip
WHERE {
  ?id a sz:Person ;
    skos:prefLabel ?descrip .
}""")

df_er_person_nodes.replace_column(
    0,
    df_er_person_nodes["id"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_person_nodes = df_er_person_nodes.with_columns(pl.lit("sz:Person").alias("class"))

df_er_person_nodes

id,descrip,class
str,str,str
"""sz:1""","""Abassin Badshah""","""sz:Person"""
"""sz:10""","""Nicholas Thomas Wright""","""sz:Person"""
"""sz:100001""","""Robert Smith""","""sz:Person"""
"""sz:100005""","""Robert E Smith Sr""","""sz:Person"""
"""sz:100006""","""Eddie Kusha""","""sz:Person"""
…,…,…
"""sz:8""","""Gulnara Suleimanova KERIMOVA""","""sz:Person"""
"""sz:83""","""Søren Kurt Hansen""","""sz:Person"""
"""sz:84""","""Anders Kjærgaard Frandsen""","""sz:Person"""


Build a dataframe for the `sz:Organization` nodes from ER.

In [25]:
df_er_organ_nodes: pl.DataFrame = run_sparql("""
SELECT ?id ?descrip
WHERE {
  ?id a sz:Organization ;
    skos:prefLabel ?descrip .
}""")

df_er_organ_nodes.replace_column(
    0,
    df_er_organ_nodes["id"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)
df_er_organ_nodes = df_er_organ_nodes.with_columns(pl.lit("sz:Organization").alias("class"))


df_er_organ_nodes

id,descrip,class
str,str,str
"""sz:100""","""GREENLIGHT GROUP LIMITED""","""sz:Organization"""
"""sz:100091""","""Hajah Mamunah Jln Pisang""","""sz:Organization"""
"""sz:100094""","""Mullenkrants""","""sz:Organization"""
"""sz:100096""","""Universal Exports USA""","""sz:Organization"""
"""sz:100097""","""Universal Exports Worldwide""","""sz:Organization"""
…,…,…
"""sz:94""","""RMLM ASSOCIATES LTD""","""sz:Organization"""
"""sz:95""","""SBE U K LIMITED""","""sz:Organization"""
"""sz:97""","""WRIGHT PUBLISHING LIMITED""","""sz:Organization"""


Build a dataframe for the `sz:DataRecord` nodes from ER.

In [9]:
df_er_data_nodes: pl.DataFrame = run_sparql("""
SELECT ?rec_iri ?rec_key ?data_src
WHERE {
  ?rec_iri a sz:DataRecord ;
    dc:identifier ?rec_key ;
    prov:wasQuotedFrom ?data_src .
}""")

df_er_data_nodes.replace_column(
    0,
    df_er_data_nodes["rec_iri"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_data_nodes.replace_column(
    2,
    df_er_data_nodes["data_src"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_data_nodes.insert_column(
    0,
    pl.Series("node_id", map(get_node_id, df_er_data_nodes["rec_iri"])),
)

df_er_data_nodes

node_id,rec_iri,rec_key,data_src
i64,str,str,str
289,"""sz:ds_customers_1001""","""1001""","""sz:ds_customers"""
290,"""sz:ds_customers_1002""","""1002""","""sz:ds_customers"""
291,"""sz:ds_customers_1003""","""1003""","""sz:ds_customers"""
292,"""sz:ds_customers_1004""","""1004""","""sz:ds_customers"""
293,"""sz:ds_customers_1005""","""1005""","""sz:ds_customers"""
…,…,…,…
725,"""sz:ds_watchlist_1042""","""1042""","""sz:ds_watchlist"""
726,"""sz:ds_watchlist_2052""","""2052""","""sz:ds_watchlist"""
727,"""sz:ds_watchlist_2062""","""2062""","""sz:ds_watchlist"""


Build a dataframe for the relations among entities and source data records.

In [10]:
df_er_rel_nodes: pl.DataFrame = run_sparql("""
SELECT ?ent ?rel_ent ?sem_rel ?why ?evidence
WHERE {
  ?bl rdf:predicate ?sem_rel ;
    rdf:subject ?ent ;
    rdf:object ?rel_ent ;
    sz:match_level ?why ;
    sz:match_key ?evidence .
}""")

df_er_rel_nodes.replace_column(
    0,
    df_er_rel_nodes["ent"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_rel_nodes.replace_column(
    1,
    df_er_rel_nodes["rel_ent"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_rel_nodes.replace_column(
    2,
    df_er_rel_nodes["rel_ent"].str.replace("http://www.w3.org/2004/02/skos/core#", "skos:"),
)

df_er_rel_nodes

node_id,rel_id,ent,rel_ent,rel_ent,why,evidence
i64,i64,str,str,str,str,str
192,263,"""sz:152""","""sz:52""","""sz:52""","""DISCLOSED""","""+OOR(APPOINTMENT_OF_BOARD,SHAR…"
111,210,"""sz:213""","""sz:188""","""sz:188""","""DISCLOSED""","""+OOR(:OTHER_INFLUENCE_OR_CONTR…"
282,237,"""sz:89""","""sz:247""","""sz:247""","""POSSIBLY_RELATED""","""+ADDRESS+REGISTRATION_COUNTRY-…"
27,330,"""sz:100040""","""sz:ds_customers_1056""","""sz:ds_customers_1056""","""RESOLVED""","""+DOB+DRLIC+PNAME"""
164,583,"""sz:105""","""sz:ds_open-ownership_434999453…","""sz:ds_open-ownership_434999453…","""RESOLVED""","""+NAME+ADDRESS+NATIONAL_ID+REGI…"
…,…,…,…,…,…,…
270,443,"""sz:59""","""sz:ds_open-ownership_113581144…","""sz:ds_open-ownership_113581144…","""INITIAL""","""INITIAL"""
79,22,"""sz:100132""","""sz:100030""","""sz:100030""","""POSSIBLY_RELATED""","""+SURNAME+ADDRESS"""
97,516,"""sz:132""","""sz:ds_open-ownership_163099107…","""sz:ds_open-ownership_163099107…","""INITIAL""","""INITIAL"""


## KùzuDB tables

KùzuDB is an embedded, open source graph database that supports the Cypher query language. It uses a _structured property graph_ model, which is similar to the _labeled property graph_ model you may be familiar with from other systems. The only difference being that KùzuDB requires strict data types for properties in the schema.

The following steps will create the graph schema in (node and relationship tables) and populate data into them.

In [30]:
DB_PATH: str = "./db"
shutil.rmtree(DB_PATH, ignore_errors = True)

db: kuzu.Database = kuzu.Database(DB_PATH)
conn: kuzu.Connection = kuzu.Connection(db)

In [32]:
conn.execute("DROP TABLE IF EXISTS Entity");
conn.execute("CREATE NODE TABLE IF NOT EXISTS Entity (id STRING PRIMARY KEY, descrip STRING, class STRING)");

In [33]:
df_taxo_nodes

id,descrip,class
str,str,str
"""sz:DataRecord""","""Data Record is a composite dat…","""skos:Concept"""
"""sz:Entity""","""Entity is anything that can be…","""skos:Concept"""
"""sz:Organization""","""Organization is a social entit…","""skos:Concept"""
"""sz:Person""","""Person represents an individua…","""skos:Concept"""


In [35]:
conn.execute("COPY Entity FROM (LOAD FROM df_taxo_nodes RETURN id, descrip, class)");

In [36]:
conn.execute("COPY Entity FROM (LOAD FROM df_er_person_nodes RETURN id, descrip, class)");
conn.execute("COPY Entity FROM (LOAD FROM df_er_organ_nodes RETURN id, descrip, class)");

In [37]:
# For visualizing the Kuzu graph in yFiles widget
from yfiles_jupyter_graphs_for_kuzu import KuzuGraphWidget

ModuleNotFoundError: No module named 'yfiles_jupyter_graphs_for_kuzu'